# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshit5445/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

## Lane 2 — Refresh / Content Opportunity Scoring

The goal is to rank content items for refresh/review priority. The model is decision-support for a content/SEO reviewer; it does not claim that a refresh will recover traffic.

This notebook uses February 2026 information as the decision-time feature window and March 2026 as the outcome window. The model and the reconstructed Week-4 baseline are evaluated on the same held-out client groups using Precision@50.

## 1. Method choice and why

I use a **Random Forest classifier** because the lane is a ranking problem with a binary outcome proxy (`went_dark`). The feature set is small and numeric, and a tree ensemble can capture non-linear relationships between visibility, CTR, position, and data availability without requiring a linear relationship.

The model produces a probability score. Pages are ranked by that score, and the same Precision@50 metric is used for the model and the Week-4 rule.

The baseline is deliberately kept simple: position-relative CTR opportunity plus impression volume. The model only earns a useful result if it beats that baseline on the same held-out data.

In [1]:
# Imports and reproducible settings
import os
import duckdb
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import precision_score, recall_score, roc_auc_score

RANDOM_STATE = 42
TOP_K = 50

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN is required. Store it in Colab Secrets; do not paste it into the notebook.")

con = duckdb.connect()
con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")
con.execute("SET VARIABLE hf_token = ?", [HF_TOKEN])
con.execute("""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN getvariable('hf_token')
)
""")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = REL + "/fact_content_daily_performance"
FEB = "read_parquet('" + FACT + "/month=2026-02/*.parquet')"
MAR = "read_parquet('" + FACT + "/month=2026-03/*.parquet')"

print("Connection ready.")

Connection ready.


## Build the model dataset

The features are restricted to February 2026. March is used only to create the outcome proxy. Identifiers are retained only for grouping and joining, then excluded from `X`.

In [2]:
# February feature frame
feb_features = con.sql(f"""
WITH feb AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(CASE WHEN gsc_data_available IS TRUE
                 THEN gsc_impressions ELSE 0 END) AS impressions_feb,
        SUM(CASE WHEN gsc_data_available IS TRUE
                 THEN gsc_clicks ELSE 0 END) AS clicks_feb,
        SUM(CASE WHEN gsc_data_available IS TRUE
                 THEN gsc_sum_position ELSE 0 END)
        / NULLIF(
            SUM(CASE WHEN gsc_data_available IS TRUE
                     THEN gsc_impressions ELSE 0 END), 0
        ) AS avg_position_feb,
        SUM(CASE WHEN ga4_data_available IS TRUE
                 THEN ga4_sessions ELSE 0 END) AS ga4_sessions_feb,
        MAX(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END)
            AS gsc_available_feb,
        MAX(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END)
            AS ga4_available_feb
    FROM {FEB}
    GROUP BY 1, 2
)
SELECT *,
       CASE
           WHEN impressions_feb > 0
           THEN clicks_feb / impressions_feb
           ELSE 0
       END AS ctr_feb
FROM feb
""").df()

# March outcome proxy
march_labels = con.sql(f"""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(CASE WHEN gsc_data_available IS TRUE
                 THEN gsc_clicks ELSE 0 END) AS clicks_mar,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS measured_gsc_days_mar
    FROM {MAR}
    GROUP BY 1, 2
)
SELECT
    client_hash_id,
    content_hash_id,
    clicks_mar,
    measured_gsc_days_mar,
    CASE WHEN clicks_mar = 0 THEN 1 ELSE 0 END AS went_dark
FROM march
WHERE measured_gsc_days_mar > 0
""").df()

frame = feb_features.merge(
    march_labels,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

# Fill feature missingness using only the February feature panel.
if frame["avg_position_feb"].isna().any():
    median_position = feb_features["avg_position_feb"].median()
    frame["avg_position_feb"] = frame["avg_position_feb"].fillna(median_position)

frame["ga4_sessions_feb"] = frame["ga4_sessions_feb"].fillna(0)

feature_columns = [
    "impressions_feb",
    "clicks_feb",
    "ctr_feb",
    "avg_position_feb",
    "ga4_sessions_feb",
    "gsc_available_feb",
    "ga4_available_feb",
]

X = frame[feature_columns].copy()
y = frame["went_dark"].astype(int)
groups = frame["client_hash_id"]

print("Rows:", len(frame))
print("Features:", feature_columns)
print("Positive labels:", int(y.sum()))
print("Missing feature values:", int(X.isna().sum().sum()))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 161539
Features: ['impressions_feb', 'clicks_feb', 'ctr_feb', 'avg_position_feb', 'ga4_sessions_feb', 'gsc_available_feb', 'ga4_available_feb']
Positive labels: 98368
Missing feature values: 0


## 2. Split design

I use a **grouped train/test split by client**. A client is kept entirely in either train or test so that the model is evaluated on clients it did not see during training.

This is a conservative validation design for this lane because pages from the same client can share measurement and content patterns. The split is not a time split because the model is already defined around a fixed February decision window and March outcome window; the important leakage boundary is that March outcome fields are never features.

In [3]:
# Grouped 80/20 client split
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE
)

train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()
y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])

assert train_clients.isdisjoint(test_clients)

print("Train rows:", len(train_idx))
print("Test rows:", len(test_idx))
print("Train clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Client overlap:", len(train_clients.intersection(test_clients)))
print("Train positive rate:", round(y_train.mean(), 4))
print("Test positive rate:", round(y_test.mean(), 4))

Train rows: 105652
Test rows: 55887
Train clients: 33
Test clients: 9
Client overlap: 0
Train positive rate: 0.5898
Test positive rate: 0.6451


## 3. Train + compare vs my baseline

Both approaches are evaluated on exactly the same held-out test rows with **Precision@50**.

The Week-4 baseline is reconstructed on the same February decision-time features. It uses the same rule logic: CTR below the observed mean for the page's position bucket creates CTR opportunity, combined with normalized impression volume.

No March outcome field is used by either ranking rule.

In [4]:
# Train the Random Forest
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=20,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

model.fit(X_train, y_train)

model_score = model.predict_proba(X_test)[:, 1]

# Reconstruct the Week-4 baseline on the same test rows.
baseline_test = frame.iloc[test_idx].copy()

def make_position_bucket(position):
    if position <= 3:
        return "top_3"
    elif position <= 10:
        return "page_1"
    elif position <= 20:
        return "page_2"
    elif position <= 50:
        return "page_3_5"
    else:
        return "deep"

# Compute position benchmarks using TRAINING rows only.
# This prevents test-set information from entering the baseline benchmark.
train_for_baseline = frame.iloc[train_idx].copy()
train_for_baseline["position_bucket"] = train_for_baseline[
    "avg_position_feb"
].apply(make_position_bucket)

benchmark = (
    train_for_baseline
    .groupby("position_bucket")["ctr_feb"]
    .mean()
)

baseline_test["position_bucket"] = baseline_test[
    "avg_position_feb"
].apply(make_position_bucket)

baseline_test["benchmark_ctr"] = (
    baseline_test["position_bucket"].map(benchmark)
)

baseline_test["ctr_gap"] = (
    baseline_test["benchmark_ctr"] - baseline_test["ctr_feb"]
).clip(lower=0)

max_gap = train_for_baseline.assign(
    position_bucket=train_for_baseline["avg_position_feb"].apply(make_position_bucket)
).assign(
    benchmark_ctr=lambda d: d["position_bucket"].map(benchmark)
).assign(
    ctr_gap=lambda d: (d["benchmark_ctr"] - d["ctr_feb"]).clip(lower=0)
)["ctr_gap"].max()

max_impressions = train_for_baseline["impressions_feb"].max()

baseline_test["ctr_opportunity_score"] = (
    baseline_test["ctr_gap"] / max_gap
    if max_gap > 0 else 0.0
)

baseline_test["volume_score"] = (
    baseline_test["impressions_feb"] / max_impressions
    if max_impressions > 0 else 0.0
)

baseline_score = 100 * (
    0.70 * baseline_test["ctr_opportunity_score"]
    + 0.30 * baseline_test["volume_score"]
)

def precision_at_k(y_true, scores, k=50):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)
    order = np.argsort(-scores, kind="mergesort")[:k]
    return float(y_true[order].mean())

baseline_p50 = precision_at_k(y_test, baseline_score, TOP_K)
model_p50 = precision_at_k(y_test, model_score, TOP_K)

# Useful secondary metrics at a fixed 0.5 threshold.
model_pred = (model_score >= 0.5).astype(int)

roc_auc = roc_auc_score(y_test, model_score)
model_precision = precision_score(y_test, model_pred, zero_division=0)
model_recall = recall_score(y_test, model_pred, zero_division=0)

comparison = pd.DataFrame({
    "method": ["Week-4 baseline", "Random Forest"],
    "precision_at_50": [baseline_p50, model_p50],
    "roc_auc": [np.nan, roc_auc],
    "precision_at_0.5": [np.nan, model_precision],
    "recall_at_0.5": [np.nan, model_recall],
})

display(comparison)

print(f"Baseline Precision@50: {baseline_p50:.3f}")
print(f"Random Forest Precision@50: {model_p50:.3f}")
print(f"Random Forest ROC-AUC: {roc_auc:.3f}")

if model_p50 > baseline_p50:
    print("Observed result: the model has higher Precision@50 than the baseline on this held-out split.")
elif model_p50 < baseline_p50:
    print("Observed result: the baseline has higher Precision@50 than the model on this held-out split.")
else:
    print("Observed result: the model and baseline have the same Precision@50 on this held-out split.")

,method,precision_at_50,roc_auc,precision_at_0.5,recall_at_0.5
0,Week-4 baseline,0.38,NaN,NaN,NaN
1,Random Forest,0.98,0.846057,0.821562,0.895349


Baseline Precision@50: 0.380
Random Forest Precision@50: 0.980
Random Forest ROC-AUC: 0.846
Observed result: the model has higher Precision@50 than the baseline on this held-out split.


## 4. Errors and interpretation

The top-ranked model results are useful only if the errors are understood. I inspect false positives among the model's top 50 and compare feature values for top-ranked positives and negatives.

Feature importance is treated as an interpretation aid, not as evidence of causality.

In [5]:
# Top-50 error inspection
test_view = frame.iloc[test_idx].copy()
test_view["model_score"] = model_score
test_view["baseline_score"] = baseline_score
test_view["prediction_at_0_5"] = model_pred

top50_idx = np.argsort(-model_score, kind="mergesort")[:TOP_K]
top50 = test_view.iloc[top50_idx].copy()

false_positives_top50 = top50[top50["went_dark"] == 0].copy()
true_positives_top50 = top50[top50["went_dark"] == 1].copy()

print("Top-50 true positives:", len(true_positives_top50))
print("Top-50 false positives:", len(false_positives_top50))

error_columns = [
    "went_dark",
    "model_score",
    "impressions_feb",
    "ctr_feb",
    "avg_position_feb",
    "ga4_sessions_feb",
    "gsc_available_feb",
    "ga4_available_feb",
]

display(false_positives_top50[error_columns].head(10))

# Feature importance
importance = pd.DataFrame({
    "feature": feature_columns,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

display(importance)

# Simple comparison of top-50 positives vs negatives
summary = pd.DataFrame({
    "top50_model_score_mean": top50.groupby("went_dark")["model_score"].mean(),
    "top50_impressions_mean": top50.groupby("went_dark")["impressions_feb"].mean(),
    "top50_ctr_mean": top50.groupby("went_dark")["ctr_feb"].mean(),
    "top50_position_mean": top50.groupby("went_dark")["avg_position_feb"].mean(),
})

display(summary)

print(
    "Interpretation: the model is a ranking aid. False positives show cases "
    "where the February signals resemble pages that later went dark even though "
    "the observed March outcome was non-dark. Feature importance describes model "
    "reliance on the available features; it does not establish causality."
)

Top-50 true positives: 49
Top-50 false positives: 1


,went_dark,model_score,impressions_feb,ctr_feb,avg_position_feb,ga4_sessions_feb,gsc_available_feb,ga4_available_feb
37371,0,0.947726,4.0,0.0,65.5,0.0,1,0


,feature,importance
0,impressions_feb,0.418197
1,clicks_feb,0.289182
2,ctr_feb,0.217297
4,ga4_sessions_feb,0.033862
3,avg_position_feb,0.022915
5,gsc_available_feb,0.013943
6,ga4_available_feb,0.004605


,top50_model_score_mean,top50_impressions_mean,top50_ctr_mean,top50_position_mean
went_dark,,,,
0,0.947726,4.000000,0.0,65.500000
1,0.949052,5.204082,0.0,77.925441


Interpretation: the model is a ranking aid. False positives show cases where the February signals resemble pages that later went dark even though the observed March outcome was non-dark. Feature importance describes model reliance on the available features; it does not establish causality.


## Self-check

- Every section above is filled with markdown reasoning and supporting code.
- The notebook should run top to bottom with no errors.
- The model and baseline use the same held-out client split and Precision@50.
- March outcome information is used only as the label, never as a feature.
- No client names, URLs, or private queries are included.
- Claims use careful words such as observed, measured, directional, and decision-support.
- Commit this notebook under `work/notebooks/w05_model.ipynb` after a successful Run all.